<a href="https://colab.research.google.com/github/harprabhnanda01/week_4_submission/blob/main/Another_copy_of_Module_4_Graded_Assignment_OpenRouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Tool‑Augmented LLM App (Weather + Calculator)

**Goal:** Build a minimal GenAI pipeline that uses **memory/caching** and **two tools** (weather + calculator), with a simple agent that routes to tools when needed.



## Instructions

- Implement the required functions in the **solution cells**
- Use a module‑level cache created by `setup_cache()`; repeated `(city, unit)` queries should hit the cache (no re‑fetch).
- Keep your code deterministic for grading.
- You may integrate LangChain locally, but the autograder focuses on tool behavior + caching and a minimal agent router.
- **Do not rename** required functions or change their signatures.

### Required functions

1. `setup_cache() -> Any`
2. `get_weather(city: str, unit: str = "C", fetcher: callable | None = None) -> dict`
3. `multiply(a: float, b: float) -> float`
4. `tool_registry() -> dict`
5. `agent_route(prompt: str) -> str`  (return `"weather"`, `"multiply"`, or `"none"`)
6. `agent_answer(prompt: str, tools: dict | None = None, cache=None) -> str`

### Playtests (not graded)
Try these prompts after you finish:
- “What’s the weather in Paris?”
- “What’s 42 × 19?”
- “Summarize both results in one sentence.”


In [2]:

# Read-Only imports (you may import more inside solutions)
from typing import Any, Callable, Dict, Optional
import re
import requests  # allowed for tests


## OpenRouter API Setup

This assignment is configured to use **OpenRouter** instead of provider-specific API keys. Enter your OpenRouter API key when prompted. Do not hard-code the key or save it in the notebook.

The starter assignment and grading logic focus on the required tool, caching, and routing functions. The OpenRouter configuration is provided so any optional LLM calls can use the same key.

In [1]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key configured.")


Enter your OpenRouter API key: ··········
OpenRouter API key configured.



### Q1. Implement `setup_cache()` *(Solution cell)*
Create a cache object (e.g., dict) and assign it to module-level `CACHE`. Return it.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [3]:
from typing import Any

CACHE: Any = None

def setup_cache() -> Any:
    """Initialize and assign a cache object to module-level CACHE."""
    global CACHE
    CACHE = {}
    return CACHE

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `setup_cache`.
 -  If the function is not found, you answer will be rejected by the system

In [4]:
# BEGIN PUBLIC TESTS
c = setup_cache()
assert c is not None and isinstance(c, dict)
assert 'CACHE' in globals() and isinstance(CACHE, dict)
# END PUBLIC TESTS

In [5]:
CACHE: Any = None

def setup_cache() -> Any:
    """Initialize and assign a cache object to module-level CACHE."""
    global CACHE
    CACHE = {}
    return CACHE

In [6]:
from typing import Any, Callable, Dict, Optional
import requests
import re
import os


### Q2. Implement `get_weather(city, unit="C")` *(Solution cell)*
- Use Open-Meteo Geocoding to resolve city → (lat, lon).
- Call Open-Meteo Current Weather for temperature (Celsius).
- If `unit == "F"`, convert Celsius to Fahrenheit.
- Cache result dict as `{"city": city, "temp": <float>, "unit": unit}` under normalized key.
- Subsequent calls with same `(city, unit)` must return the **same dict object** from cache (so `is` equality works).


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [7]:

def get_weather(city: str, unit: str = "C",
                fetcher: Optional[Callable] = None) -> dict:

    global CACHE

    if CACHE is None:
        setup_cache()

    city = city.strip()
    unit = unit.strip().upper()

    if not city:
        raise ValueError("City cannot be empty.")

    if unit not in ("C", "F"):
        raise ValueError("Unit must be C or F.")

    cache_key = (city.casefold(), unit)

    if cache_key in CACHE:
        return CACHE[cache_key]

    if fetcher is not None:
        try:
            fetched = fetcher(city, unit)
        except TypeError:
            fetched = fetcher(city)

        if isinstance(fetched, dict):
            temp = float(fetched["temp"])
            fetched_unit = fetched.get("unit", "C").upper()

            if fetched_unit != unit:
                if fetched_unit == "C":
                    temp = (temp * 9 / 5) + 32
                else:
                    temp = (temp - 32) * 5 / 9
        else:
            temp = float(fetched)
            if unit == "F":
                temp = (temp * 9 / 5) + 32

    else:
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"

        geo_response = requests.get(
            geo_url,
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=15
        )
        geo_response.raise_for_status()

        geo_data = geo_response.json()
        results = geo_data.get("results", [])

        if not results:
            raise ValueError(f"City not found: {city}")

        latitude = results[0]["latitude"]
        longitude = results[0]["longitude"]

        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m"
            },
            timeout=15
        )
        weather_response.raise_for_status()

        weather_data = weather_response.json()
        temp = float(weather_data["current"]["temperature_2m"])

        if unit == "F":
            temp = (temp * 9 / 5) + 32

    result = {
        "city": city,
        "temp": float(temp),
        "unit": unit
    }

    CACHE[cache_key] = result
    return result


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `get_weather`.
 -  If the function is not found, you answer will be rejected by the system

In [8]:
# Q2 — Public Tests (API call allowed)
# BEGIN PUBLIC TESTS
setup_cache()
d1 = get_weather("Paris", "C")
d2 = get_weather("paris", "c")
assert d1 is d2, "Cached object should be returned for repeated queries."
assert isinstance(d1["temp"], float) and d1["unit"] == "C"
# Fahrenheit conversion
d3 = get_weather("Paris", "F")
assert d3["unit"] == "F" and isinstance(d3["temp"], float)
# END PUBLIC TESTS


### Q3. Implement `multiply(a, b)` *(Solution cell)*
Return the product of `a` and `b`.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [9]:

def multiply(a: float, b: float) -> float:
    return float(a) * float(b)


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `multiply`.
 -  If the function is not found, you answer will be rejected by the system

In [10]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert multiply(6, 7) == 42
assert abs(multiply(1.5, 2.0) - 3.0) < 1e-9
# END PUBLIC TESTS

In [11]:
# Q3 — Hidden Tests


### Q4. Implement `tool_registry()` *(Solution cell)*
Return a dict mapping `{"weather": get_weather, "multiply": multiply}`.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [12]:

def tool_registry() -> Dict[str, Callable]:
    return {
        "weather": get_weather,
        "multiply": multiply
    }


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `tool_registry`.
 -  If the function is not found, you answer will be rejected by the system

In [13]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
reg = tool_registry()
assert isinstance(reg, dict)
assert "weather" in reg and "multiply" in reg
assert callable(reg["weather"]) and callable(reg["multiply"])
# END PUBLIC TESTS

In [14]:
# Q4 — Hidden Tests


### Q5. Implement `agent_route(prompt)` *(Solution cell)*
Return `"weather"`, `"multiply"`, or `"none"` based on the prompt.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [15]:

def agent_route(prompt: str) -> str:

    prompt = prompt.lower()

    weather_words = [
        "weather", "temperature", "forecast",
        "hot", "cold", "rain", "climate"
    ]

    if any(word in prompt for word in weather_words):
        return "weather"

    multiplication_words = [
        "multiply", "multiplication", "product"
    ]

    has_multiply_symbol = bool(
        re.search(
            r"\d+(?:\.\d+)?\s*[x×*]\s*\d+(?:\.\d+)?",
            prompt
        )
    )

    if any(word in prompt for word in multiplication_words):
        return "multiply"

    if has_multiply_symbol:
        return "multiply"

    return "none"


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `agent_route`.
 -  If the function is not found, you answer will be rejected by the system

In [16]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
assert agent_route("what's the weather in Jaipur?") == "weather"
assert agent_route("compute 42 x 19 please") == "multiply"
assert agent_route("say hello") == "none"
# END PUBLIC TESTS


In [17]:
# Q5 — Hidden Tests


### Q6. Implement `agent_answer(prompt, tools=None, cache=None)` *(Solution cell)*
- Use `agent_route` to choose a tool.
- If weather: parse city from `"weather in <City>"`. Default unit `"C"`, but if prompt mentions "Fahrenheit" or "F", use `"F"`.
- If multiply: parse two numbers from `a x b` or `a * b` and compute result with `multiply`.
- Return a short string response. You may pass `cache` in to override the module cache.


### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [18]:

def agent_answer(prompt: str,
                 tools: dict = None,
                 cache=None) -> str:

    selected_tool = agent_route(prompt)

    if tools is None:
        tools = tool_registry()

    global CACHE
    original_cache = CACHE

    if cache is not None:
        CACHE = cache
    elif CACHE is None:
        setup_cache()

    try:
        if selected_tool == "weather":

            match = re.search(
                r"(?:weather|temperature|forecast)\s+(?:in|for)\s+([a-zA-Z\s]+)",
                prompt,
                re.IGNORECASE
            )

            if not match:
                return "Please specify a city."

            city = match.group(1).strip()

            city = re.split(
                r"\s+(?:in|at|with|and)\s+",
                city,
                maxsplit=1,
                flags=re.IGNORECASE
            )[0].strip()

            unit = "F" if re.search(
                r"\b(fahrenheit|in\s+F)\b",
                prompt,
                re.IGNORECASE
            ) else "C"

            result = tools["weather"](city, unit)

            return (
                f"The current temperature in {result['city']} "
                f"is {result['temp']:.1f}°{result['unit']}."
            )

        elif selected_tool == "multiply":

            match = re.search(
                r"(-?\d+(?:\.\d+)?)\s*[x×*]\s*(-?\d+(?:\.\d+)?)",
                prompt
            )

            if match:
                a = float(match.group(1))
                b = float(match.group(2))
            else:
                numbers = re.findall(
                    r"-?\d+(?:\.\d+)?",
                    prompt
                )

                if len(numbers) < 2:
                    return "Please provide two numbers."

                a, b = float(numbers[0]), float(numbers[1])

            answer = tools["multiply"](a, b)

            return f"{a:g} * {b:g} = {answer:g}"

        else:
            return (
                "No external tool is required for this query. "
                "Please ask a weather or multiplication question."
            )

    finally:
        CACHE = original_cache


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `agent_answer`.
 -  If the function is not found, you answer will be rejected by the system

In [19]:

# Q6 — Public Tests
# BEGIN PUBLIC TESTS
cache = setup_cache()
tools = tool_registry()

o1 = agent_answer("What's the weather in Mumbai in Fahrenheit?", tools=tools, cache=cache)
assert "Mumbai" in o1 and ("F" in o1)

o2 = agent_answer("Please do 12 * 8", tools=tools, cache=cache)
assert "12.0 * 8.0 = 96.0" in o2 or "12 * 8 = 96" in o2
# END PUBLIC TESTS


In [20]:

# Q6 — Hidden Tests


In [21]:

!pip -q install -U langchain langchain-openai langchain-core


In [22]:

import os
import time
import json

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

# Use the OpenRouter key already configured in the notebook
assert os.environ.get("OPENROUTER_API_KEY"), "Please configure your OpenRouter API key."

# Enable LangChain's in-memory LLM response cache
set_llm_cache(InMemoryCache())

# Connect LangChain to OpenRouter
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

print("LangChain and OpenRouter configured successfully!")


LangChain and OpenRouter configured successfully!


In [23]:

@tool
def weather_tool(city: str, unit: str = "C") -> str:
    """Get the current weather temperature for a city in Celsius or Fahrenheit."""
    result = get_weather(city, unit)
    return (
        f"Current temperature in {result['city']} "
        f"is {result['temp']:.1f} degrees {result['unit']}."
    )


@tool
def calculator_tool(a: float, b: float) -> str:
    """Multiply two numbers and return the exact result."""
    result = multiply(a, b)
    return f"{a} multiplied by {b} equals {result}."


langchain_tools = [weather_tool, calculator_tool]

tool_lookup = {
    "weather_tool": weather_tool,
    "calculator_tool": calculator_tool
}

print("Weather and Calculator tools registered!")


Weather and Calculator tools registered!


In [24]:

# Give the tools to the LLM
agent_llm = llm.bind_tools(langchain_tools)


def run_langchain_agent(question, max_steps=4):

    messages = [
        HumanMessage(content=question)
    ]

    for step in range(max_steps):

        response = agent_llm.invoke(messages)
        messages.append(response)

        # If the model does not request a tool, return its answer.
        if not response.tool_calls:
            return response.content

        # Execute the requested tools.
        for call in response.tool_calls:

            tool_name = call["name"]
            tool_args = call["args"]

            if tool_name not in tool_lookup:
                result = "Requested tool is unavailable."
            else:
                result = tool_lookup[tool_name].invoke(tool_args)

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"]
                )
            )

    return "Maximum tool steps reached."


print("LangChain agent workflow created!")


LangChain agent workflow created!


In [25]:

comparison_questions = [
    "What's the current weather in Paris?",
    "What's 42 multiplied by 19?",
]

comparison_results = []

# Run first two queries
for question in comparison_questions:

    print("\n" + "=" * 60)
    print("QUESTION:", question)

    # Vanilla LLM
    start = time.time()
    vanilla_response = llm.invoke(question)
    vanilla_time = time.time() - start

    # Tool-augmented LangChain agent
    start = time.time()
    agent_response = run_langchain_agent(question)
    agent_time = time.time() - start

    comparison_results.append({
        "question": question,
        "vanilla": vanilla_response.content,
        "agent": agent_response,
        "vanilla_time": round(vanilla_time, 2),
        "agent_time": round(agent_time, 2)
    })

    print("\nVANILLA LLM:")
    print(vanilla_response.content)

    print("\nTOOL-AUGMENTED LLM:")
    print(agent_response)

    print("\nVanilla time:", round(vanilla_time, 2), "seconds")
    print("Agent time:", round(agent_time, 2), "seconds")


# Third query: summarize both results
weather_result = comparison_results[0]["agent"]
calculation_result = comparison_results[1]["agent"]

summary_question = (
    "Summarize these two results in one sentence: "
    + weather_result + " "
    + calculation_result
)

print("\n" + "=" * 60)
print("QUESTION 3: Summarize both results in one sentence.")

start = time.time()
vanilla_summary = llm.invoke(summary_question)
vanilla_summary_time = time.time() - start

start = time.time()
agent_summary = run_langchain_agent(summary_question)
agent_summary_time = time.time() - start

comparison_results.append({
    "question": "Summarize both results in one sentence.",
    "vanilla": vanilla_summary.content,
    "agent": agent_summary,
    "vanilla_time": round(vanilla_summary_time, 2),
    "agent_time": round(agent_summary_time, 2)
})

print("\nVANILLA LLM:")
print(vanilla_summary.content)

print("\nTOOL-AUGMENTED LLM:")
print(agent_summary)

print("\nAll three comparisons completed!")


QUESTION: What's the current weather in Paris?

VANILLA LLM:
I'm unable to provide real-time weather updates. To get the current weather in Paris, I recommend checking a reliable weather website or app.

TOOL-AUGMENTED LLM:
The current temperature in Paris is 12.5 degrees Celsius.

Vanilla time: 1.74 seconds
Agent time: 1.98 seconds

QUESTION: What's 42 multiplied by 19?

VANILLA LLM:
42 multiplied by 19 equals 798.

TOOL-AUGMENTED LLM:
42 multiplied by 19 equals 798.

Vanilla time: 0.73 seconds
Agent time: 1.81 seconds

QUESTION 3: Summarize both results in one sentence.

VANILLA LLM:
The current temperature in Paris is 12.5 degrees Celsius, and 42 multiplied by 19 equals 798.

TOOL-AUGMENTED LLM:
The current temperature in Paris is 12.5 degrees Celsius, and 42 multiplied by 19 equals 798.

All three comparisons completed!


In [26]:

cache_question = "Explain in one sentence what a calculator tool does."

# First request
start = time.time()
first_response = llm.invoke(cache_question)
first_time = time.time() - start

# Repeat exactly the same request
start = time.time()
second_response = llm.invoke(cache_question)
second_time = time.time() - start

print("First response:", first_response.content)
print("Second response:", second_response.content)

print("\nFirst call time:", round(first_time, 3), "seconds")
print("Cached call time:", round(second_time, 3), "seconds")

print("\nLangChain InMemoryCache test completed!")


First response: A calculator tool performs mathematical calculations, allowing users to input numbers and operations to obtain results.
Second response: A calculator tool performs mathematical calculations, allowing users to input numbers and operations to obtain results.

First call time: 0.851 seconds
Cached call time: 0.001 seconds

LangChain InMemoryCache test completed!


# Reflection: Building a Tool-Augmented LLM Application

The objective of this assignment was to build a tool-augmented Generative AI application using an LLM, external tools, memory, and caching. I implemented a weather tool and a calculator tool and connected them to a language model through LangChain and OpenRouter. The application demonstrates how an LLM can use external functions to perform tasks that go beyond ordinary text generation.

### 1. How tool use extended the LLM's capabilities

A traditional LLM generates responses using its learned knowledge and the information provided in the prompt. However, it may not always have access to current information or produce reliable results for every calculation. External tools help overcome these limitations.

In this application, I implemented a weather tool using the Open-Meteo API, which retrieves current temperature information for a requested city. I also implemented a calculator tool using Python to perform multiplication accurately. LangChain tools were exposed to the model, allowing it to identify when an external function was needed.

For example, when a user asks about the weather in Paris, the model can select the weather tool. When a user asks for the result of 42 multiplied by 19, it can select the calculator tool. This makes the application more practical because it can retrieve information and perform operations instead of relying entirely on generated responses.

### 2. Role of memory and caching

Caching improves application efficiency by storing previously generated responses and retrieved information. I implemented a dictionary-based cache for weather results and LangChain's InMemoryCache for repeated LLM requests.

The weather cache stores results using the city and temperature unit as the key. When the same city and unit are requested again, the application can return the stored result without making another API request. Similarly, LangChain's InMemoryCache can reuse responses for identical model requests.

This reduces unnecessary network calls, improves response time for repeated queries, and helps reduce API usage. However, cached weather information can become outdated, so an expiry mechanism would be useful in a real-world application.

### 3. Trade-offs in speed, complexity, and transparency

Tool augmentation improves functionality and reliability for tasks requiring external information or accurate calculations. However, it also introduces additional complexity because the application needs tool definitions, routing, API connections, caching, and error handling.

A weather request depends on an internet connection and the availability of the external API. Therefore, it may take longer than a direct LLM response. A repeated request can become faster because the result may already be cached. The calculator tool is comparatively simple because it performs calculations locally.

Another advantage is transparency. The tool workflow makes it easier to understand which external function is being used and why. However, the application must also handle cases where the model selects an inappropriate tool or fails to provide valid arguments.

### 4. Possible improvements and extensions

The application could be extended by adding tools for news search, currency conversion, web search, and database queries. Multiple APIs could provide information from different sources and improve the application's usefulness.

A hybrid caching strategy could also be introduced. Frequently requested information could be stored for longer periods, while time-sensitive information such as weather could have a shorter cache lifetime. This would maintain a balance between response speed and information freshness.

Another improvement would be function chaining. For example, the application could retrieve weather information for multiple cities, compare their temperatures, and generate a summary. This would allow the system to complete more complex multi-step tasks.

### Conclusion

Overall, this assignment helped me understand how LLMs, external tools, LangChain, and caching work together. I learned that an LLM does not need to perform every task independently. By connecting it to specialised tools, an application can retrieve current information, perform accurate calculations, and reuse previous results. These concepts provide a foundation for developing more advanced and reliable AI agents.

---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. If an API key is required for optional LLM functionality, use the OpenRouter API key configured above. The required grading functions focus on tool behavior, caching, and agent routing.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
